# Classification of messages as spam or not spam using Naive Bayes algorithm

**Import Dataset - upload the SMS text file to the content folder on the left panel before running**

In [366]:
import pandas as pd
import numpy as np

# Import Dataset - upload the SMS text file to the content folder on the left panel before running
df = pd.read_table('SMS.txt', sep='\t', header=None, names=['label', 'sms_message'])
df

,label,sms_message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [367]:
# map the 'ham' value to 0 and the 'spam' value to 1.
df['label_binary'] = df.label.map({'ham':0,'spam':1})
df.head()

,label,sms_message,label_binary
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [368]:
# Get stats
df['label'].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

In [369]:
#  data cleaning
df['sms_message'] = df['sms_message'].str.replace(r'[\W_]+', ' ', regex=True).str.strip() # Removes punctuation and leading/trailing spaces
df['sms_message'] = df['sms_message'].str.lower() ### making all the words lowercase
df.head(10)

,label,sms_message,label_binary
0,ham,go until jurong point crazy available only in ...,0
1,ham,ok lar joking wif u oni,0
2,spam,free entry in 2 a wkly comp to win fa cup fina...,1
3,ham,u dun say so early hor u c already then say,0
4,ham,nah i don t think he goes to usf he lives arou...,0
5,spam,freemsg hey there darling it s been 3 week s n...,1
6,ham,even my brother is not like to speak with me t...,0
7,ham,as per your request melle melle oru minnaminun...,0
8,spam,winner as a valued network customer you have b...,1
9,spam,had your mobile 11 months or more u r entitled...,1


In [370]:
# Randomly shuffle the records in the dataset to avoid bias
df = df.sample(frac=1, random_state=1)
df.head(10)

,label,sms_message,label_binary
1078,ham,yep by the pretty sculpture,0
4028,ham,yes princess are you going to make me moan,0
958,ham,welp apparently he retired,0
4642,ham,havent,0
4674,ham,i forgot 2 ask ü all smth there s a card on da...,0
5461,ham,ok i thk i got it then u wan me 2 come now or wat,0
4210,ham,i want kfc its tuesday only buy 2 meals only 2...,0
4216,ham,no dear i was sleeping p,0
1603,ham,ok pa nothing problem,0
1504,ham,ill be there on lt gt ok,0


In [371]:
# Split into training and test sets
training_test_index = round(len(df) * 0.8)

training = df[:training_test_index].reset_index(drop=True)
test = df[training_test_index:].reset_index(drop=True)

print('-- Training set stats --')
print(training.shape)
print(training['label_binary'].value_counts())
print('-- Test set stats --')
print(test.shape)
print(test['label_binary'].value_counts())

-- Training set stats --
(4458, 3)
label_binary
0    3858
1     600
Name: count, dtype: int64
-- Test set stats --
(1114, 3)
label_binary
0    967
1    147
Name: count, dtype: int64


In [372]:
### creating vocabulary from training data
training['sms_message'] = training['sms_message'].str.split()
vocabulary = []
for sms in training['sms_message']:
   for word in sms:
      vocabulary.append(word)
vocabulary = list(set(vocabulary))  ### only count the number of unique words
print(len(vocabulary))
vocabulary[0:9]

7780


['shop',
 'ful',
 'birthday',
 'smacks',
 'memory',
 'bridge',
 'taught',
 'prabu',
 'soc']

In [373]:
word_counts_per_sms = {unique_word: [0] * len(training['sms_message']) for unique_word in vocabulary}

for index, sms in enumerate(training['sms_message']):
   for word in sms:
      word_counts_per_sms[word][index] += 1
word_counts = pd.DataFrame(word_counts_per_sms)
word_counts

,shop,ful,birthday,smacks,memory,bridge,taught,prabu,soc,notice,...,nose,balls,owns,same,visitor,subs,uve,fish,bat,yup
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4453,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4454,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4455,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4456,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [374]:
training_new = pd.concat([training, word_counts], axis=1)
training_new.head()

,label,sms_message,label_binary,shop,ful,birthday,smacks,memory,bridge,taught,...,nose,balls,owns,same,visitor,subs,uve,fish,bat,yup
0,ham,"[yep, by, the, pretty, sculpture]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,ham,"[yes, princess, are, you, going, to, make, me,...",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,ham,"[welp, apparently, he, retired]",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,ham,[havent],0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,ham,"[i, forgot, 2, ask, ü, all, smth, there, s, a,...",0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [375]:
# Run a baseline model evaluation
# Set all 'predicted to 0 or 1 randomly to get a baseline (coin-flip)
test['predicted'] = np.random.randint(0, 2, size=len(test))
test['predicted'].value_counts()

predicted
1    559
0    555
Name: count, dtype: int64

In [376]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print('Accuracy score: {}'.format(accuracy_score(test['label_binary'], test['predicted'])))
print('Precision score: {}'.format(precision_score(test['label_binary'], test['predicted'])))
print('Recall score: {}'.format(recall_score(test['label_binary'], test['predicted'])))
print('F1 score: {}'.format(f1_score(test['label_binary'], test['predicted'])))

Accuracy score: 0.5026929982046678
Precision score: 0.13595706618962433
Recall score: 0.5170068027210885
F1 score: 0.21529745042492918


# **Your implementation starts here**.


Make sure your prediction result is saved into the column `test['predicted']` for the evaluation to run automatically.  
**50 points** for successful execution of your code and producing the confusion matrix correctly

In [377]:
# Laplace smoothing
alpha = 1

In [378]:
# Hints:
# Step 1: you need to calculate P(Spam) and P(Ham)
numMessages = len(training)
numSpam = len(training[training["label_binary"] == 1])
numHam = len(training[training["label_binary"] == 0])
pSpam = numSpam / numMessages
pHam = numHam / numMessages

print(f"P(Spam) = {pSpam:.4}, p(Ham) = {pHam:.4}")

print("\n") # Seperating the sections in output

# Step 2: you need to count N_Spam, N_Ham
spamMsgs = training.loc[training["label_binary"] == 1, "sms_message"]
hamMsgs = training.loc[training["label_binary"] == 0, "sms_message"]

print(f"N_Spam = {len(spamMsgs)}, N_Ham = {len(hamMsgs)}")

print("\n") # Seperating the sections in output

# Step 3: you need to count the number of times the word w occurs in spam/ham message: N_w_spam, N_w_ham
spamWords = {}
hamWords = {}

# Go through each spam message, and keep track of them in list above
for msg in spamMsgs:
    for word in msg:
        spamWords[word] = spamWords.get(word, 0) + 1

# Go through each ham message, and keep track of them in list above
for msg in hamMsgs:
    for word in msg:
        hamWords[word] = hamWords.get(word, 0) + 1

# Determine the occurrence values, then print them out
nWSpam = sum(spamWords.values())
nWHam = sum(hamWords.values())
nVocab = len(vocabulary)

print(f"N_w_spam = {nWSpam}, N_w_ham = {nWHam}, N_Vocabulary = {nVocab}")

print("\n") # Seperating the sections in output

# Step 4: then you can calculate the prob of occurrence of each word:
#         p(w|spam)=(N_w_spam+alpha)/(N_Spam+alpha*N_Vocabulary)
#         p(w|Ham)=(N_w_ham+alpha)/(N_Ham+alpha*N_Vocabulary)   
pWGivenSpam = {}
pWGivenHam = {}

# From the equations above, calculating p(w|spam) and p(w|Ham) for each word in data
for word in vocabulary:
    pWGivenSpam[word] = (spamWords.get(word, 0) + alpha) / (nWSpam + alpha * nVocab)
    pWGivenHam[word] = (hamWords.get(word, 0) + alpha) / (nWHam + alpha * nVocab)

# Step 5: Now perform the prediction on the test dataset messages using the Naive Bayes method. Store your prediction results (1=spam or 0=ham ) to test['predicted']
predictions = []

for msg in test["sms_message"]:
    words = msg.split()

    # The starting values, what will be multiplied to determine final probabtilies and then compared 
    pSpamGivenMsg = pSpam
    pHamGivenMsg = pHam

    # Multiply P(w | class) for each word, as long as we have it in vocab
    for word in words:
        if word in pWGivenSpam:
            pSpamGivenMsg *= pWGivenSpam[word]
            pHamGivenMsg *= pWGivenHam[word]

    # Whichever probability is bigger is the one one that is more likely true, add that to predictions list
    if pSpamGivenMsg > pHamGivenMsg:
        predictions.append(1)
    else:
        predictions.append(0)

# Assign each of the various prediction models to the actual messages all at once
test["predicted"] = predictions
print(test["predicted"].value_counts())

print("\n") # Seperating the sections in output

# Step 6: Summarize the results in a confusion matrix and print out the four values of the confusion matrix
#         Verify that your printout is consistent with the output from test['label_binary'].value_counts() and test['predicted'].value_counts()
TP = 0
FP = 0
FN = 0
TN = 0

# Go through each row, compare actual vs predicted values and increment the corresponding quadrants
for i in range(len(test)):
    # Select the entire row at i using iloc for either actual/predicted values
    actual = test["label_binary"].iloc[i]
    predicted = test["predicted"].iloc[i]

    if actual == 1 and predicted == 1:
        TP += 1
    elif actual == 0 and predicted == 1:
        FP += 1
    elif actual == 1 and predicted == 0:
        FN += 1
    else:
        TN += 1

# Print out the the 4 values of the confusion matrix
print(f"TP = {TP}, FP = {FP}, FN = {FN}, TN = {TN}")

# Compare the calculated actual values to the ones that were provided in data
print(f"Actual ham (FP + TN) = {FP + TN}")
print(f"Actual spam (TP + FN) = {TP + FN}")
print(test["label_binary"].value_counts())

print("\n") # Seperating the sections in output

# Compare the predicted values to the ones that were originally from data
print(f"Predicted ham (FN + TN) = {FN + TN}")
print(f"Predicted spam (TP + FP) = {TP + FP}")
print(test["predicted"].value_counts())

P(Spam) = 0.1346, p(Ham) = 0.8654


N_Spam = 600, N_Ham = 3858


N_w_spam = 15193, N_w_ham = 57233, N_Vocabulary = 7780


predicted
0    970
1    144
Name: count, dtype: int64


TP = 139, FP = 5, FN = 8, TN = 962
Actual ham (FP + TN) = 967
Actual spam (TP + FN) = 147
label_binary
0    967
1    147
Name: count, dtype: int64


Predicted ham (FN + TN) = 970
Predicted spam (TP + FP) = 144
predicted
0    970
1    144
Name: count, dtype: int64


**Evaluate your implementation** for accuracy, precision, recall and F1_score.  The performance points of your implementation will be calculated automatically.  However, it is only awarded if the predictions are made by a Naive Bayes implementation.

**30 points** for how well your implementation predicts spam.  A correct implementation should achieve an F1 score above 0.90.  
## **DO NOT modify this cell below.**

In [379]:
# Model Evaluation
print('Accuracy score: {}'.format(accuracy_score(test['label_binary'], test['predicted'])))
print('Precision score: {}'.format(precision_score(test['label_binary'], test['predicted'])))
print('Recall score: {}'.format(recall_score(test['label_binary'], test['predicted'])))
my_f1_score = f1_score(test['label_binary'], test['predicted'])
print('F1 score: {}'.format(my_f1_score))
performance_point = round(np.clip((my_f1_score - 0.20) / (0.9-0.20) * 30, 0, 30))
print('Your performance point: {}'.format(performance_point))

Accuracy score: 0.9883303411131059
Precision score: 0.9652777777777778
Recall score: 0.9455782312925171
F1 score: 0.9553264604810997
Your performance point: 30


**Analyze your implementation of the Naive Bayes algorithm:** select an entry from each quadrant of the confusion matrix and show the details of the prediction, i.e., the probability of being a spam or a ham, and all the contributing probabilities.  Discuss why mis-classification ocurrs for the FP and FN examples.

**20 points** for a correct and clear presentation.

In [380]:
# Use the first 4 entries in each of the quadrants for the example
tpExample = test[(test["label_binary"] == 1) & (test["predicted"] == 1)].index[0]
tnExample = test[(test["label_binary"] == 0) & (test["predicted"] == 0)].index[0]
fpExample = test[(test["label_binary"] == 0) & (test["predicted"] == 1)].index[0]
fnExample = test[(test["label_binary"] == 1) & (test["predicted"] == 0)].index[0]

# The same code runs for all 4 examples, just need the actual example index and then quadrant name for the title
def PredictionDetails(example, quadrant):
    # Get the message, along with actual and predicted values, and then get the individual words from message
    msg = test.loc[example, "sms_message"]
    actual = test.loc[example, "label_binary"]
    predicted = test.loc[example, "predicted"]
    words = msg.split()

    # Provide basic info about the example message and its previously calculated info
    print(f"Example for {quadrant}")
    print(f"Message: {msg}")
    print(f"Actual: {actual}")
    print(f"Predicted: {predicted}")
    print(f"P(spam) = {pSpam:.4}, P(ham) = {pHam:.4}")

    # The starting values, what will be multiplied to determine final probabtilies and then compared 
    pSpamGivenMsg = pSpam
    pHamGivenMsg = pHam

    # Multiply P(w | class) for each word, as long as we have it in vocab
    for word in words:
        if word in pWGivenSpam:
            pSpamGivenMsg *= pWGivenSpam[word]
            pHamGivenMsg *= pWGivenHam[word]

            print(f"Word = {word}, P(w|spam) = {pWGivenSpam[word]:.4}, P(w|ham) = {pWGivenHam[word]:.4}")
    
    # Code above is reused from main implementation, tweaked to allow for individual probs and final one below
    print(f"Final P(spam | msg) = {pSpamGivenMsg}")
    print(f"Final P(ham | msg) = {pHamGivenMsg}")

    # Whichever probability is bigger is the one one that is more likely true, add that to predictions list
    if pSpamGivenMsg > pHamGivenMsg:
        print("This message is classified as spam")
    else:
        print("This message is classified as Ham")

PredictionDetails(tpExample, "TP - True Positive")

print("\n") # Seperating the sections in output

PredictionDetails(tnExample, "TN - True Negative")

print("\n") # Seperating the sections in output

PredictionDetails(fpExample, "FP - False Positive")

print("\n") # Seperating the sections in output

PredictionDetails(fnExample, "FN - False Negative")

Example for TP - True Positive
Message: had your mobile 10 mths update to latest orange camera video phones for free save s with free texts weekend calls text yes for a callback orno to opt out
Actual: 1
Predicted: 1
P(spam) = 0.1346, P(ham) = 0.8654
Word = had, P(w|spam) = 0.0004788, P(w|ham) = 0.001154
Word = your, P(w|spam) = 0.009228, P(w|ham) = 0.00523
Word = mobile, P(w|spam) = 0.004266, P(w|ham) = 0.0002153
Word = 10, P(w|spam) = 0.0009576, P(w|ham) = 0.0001846
Word = mths, P(w|spam) = 0.0001306, P(w|ham) = 3.076e-05
Word = update, P(w|spam) = 0.0004788, P(w|ham) = 7.691e-05
Word = to, P(w|spam) = 0.02381, P(w|ham) = 0.0196
Word = latest, P(w|spam) = 0.001219, P(w|ham) = 4.614e-05
Word = orange, P(w|spam) = 0.001001, P(w|ham) = 7.691e-05
Word = camera, P(w|spam) = 0.001045, P(w|ham) = 4.614e-05
Word = video, P(w|spam) = 0.001262, P(w|ham) = 4.614e-05
Word = phones, P(w|spam) = 0.0003482, P(w|ham) = 3.076e-05
Word = for, P(w|spam) = 0.006704, P(w|ham) = 0.006337
Word = free, P(w|

## Analysis / Discussion
### TP - True Positive
The text message overall was trying to advertise some kind of callback service but was caught as spam correctly, and it used a lot of keywords that had higher values of spam than ham, like "free" which had  P(w|spam) = 0.007661 and P(w|ham) = 0.0007537. There are many more similar cases for keywords in this SMS, and because of this, the final probabiliies also reflect these maginitude differneces. We ended up with having a 15 order maginutide difference, and it shows what one of the best possible cases for Naive Bayes is good for.
### TN - True Negative
For this example, a regular non spam text message was properly classified as ham. One of the biggest words in this text message was the letter i, which had P(w|spam) = 0.00222 and P(w|ham) = 0.03695. Overall, there were many terms in this text message that made it stand out as a more personal one rather than some kind of transactional one, but i was one of the largest differences. Because the ham was 17x more likely than the spam, it was one of the dominant factors in determine if it was ham, and the final probabilty for ham ends up being about 7 orders greater than spam.
### FP - False Positive
This short text message outlines one of the drawbacks with this method for detecting spam. Because there was only 4 words, and 1 of which we couldn't really compare to because we didn't have any data on it (limited). For the words that we can actual see probaitlies for, unlimited had P(w|spam) = 0.0004353 and P(w|ham) = 6.153e-05 so it was 10x more likely to be spam, and texts/minutes weren't too different. Because this was mostly leaning to spam and not ham values, it was improperly classified as spam. This shows that one of the things important for this type of algrothim is making sure there is enough context, since just these 4 words alone could be coming from a regular person, and the vocab could overlap with trained data like what happened in this exmaple.
### FN - False Negative
This text message was written in more of a personal tone (even though it's probably from a bot or something), yet it is clearly meant to be some kind of advertising. This is apparent by it having personal words like P(w|spam) = 0.001088 and P(w|ham) = 0.009844, which means its 9x more likely ham than spam, and then there's also P(w|spam) = 0.0003918 and P(w|ham) = 0.009198 which makes my be 23x more likely ham. Because of the usage of these personal words and other similar ones, it was falsely flagged as ham even though it should be spam. There were a couple of words that did lean towards spam, like call being P(w|spam) = 0.01262 and P(w|ham) = 0.003046, but overall it was more ham leaning.

**Overall, these examples show that Naive Bayes can work well when a message is properly distiributed with vocabulary that matches the training data, but there could be some edge cases that fail when we either get short messages that cause one word's values to dominate the spam/ham determination. Then there's the unexpected writing style messages, where automated messages could be written in a personal style (like the last example), where it involves things like pronouns that typically have higher ham than spam, so that will skew the overall probability.**

**Bonus for 20 points:**  Use function MultinomialNB (from sklearn.naive_bayes import MultinomialNB) to perform the same classification and evaludate its results.

In [381]:
from sklearn.naive_bayes import MultinomialNB

# The data that MultinomialNB needs to train itself (using the data that I already have provided before)
xTrain = word_counts
yTrain = training["label_binary"]

# Create the model using the data above and also the same alpha value as before
model = MultinomialNB(alpha=alpha)
model.fit(xTrain, yTrain)

# For determining the predictions
testRows = []

# Go through each message, build a list of dicts such that each text message gets its own
for msg in test["sms_message"]:
    counts = {}

    for word in msg.split():
        if word in pWGivenSpam: # Make sure the word exists in our data
            counts[word] = counts.get(word, 0) + 1

    testRows.append(counts)

# Convert back to a DataFrame, by turning the per-message dicts into a matrix, then fill anything missing with 0 and convert to ints
xTest = pd.DataFrame(testRows).reindex(columns=vocabulary, fill_value=0).fillna(0).astype(int)
yTest = test["label_binary"]

# Actually calculate the predictions, then show the new predictions vs my previously calculated version
sklearnPredictions = model.predict(xTest)

print("sklearn MultinomialNB:")
print(f"Accuracy: {accuracy_score(yTest, sklearnPredictions)}")
print(f"Precision: {precision_score(yTest, sklearnPredictions)}")
print(f"Recall: {recall_score(yTest, sklearnPredictions)}")
print(f"F1: {f1_score(yTest, sklearnPredictions)}")

print("\n") # Seperating the sections in output

print("My own detection algorithm")
print(f"Accuracy: {accuracy_score(yTest, test["predicted"])}")
print(f"Precision: {precision_score(yTest, test["predicted"])}")
print(f"Recall: {recall_score(yTest, test["predicted"])}")
print(f"F1: {f1_score(yTest, test["predicted"])}")

sklearn MultinomialNB:
Accuracy: 0.9883303411131059
Precision: 0.9652777777777778
Recall: 0.9455782312925171
F1: 0.9553264604810997


My own detection algorithm
Accuracy: 0.9883303411131059
Precision: 0.9652777777777778
Recall: 0.9455782312925171
F1: 0.9553264604810997


So, in theory, since I am using the same vocab for both methods, along with the same tokenization and also same Laplace smoothing with previous calculated probabilities for spam/ham in the first step of original implementation, we should get the same values for both methods. And indeed, after running it, we can see that both results are the exact same, and it should be true for almost all types of messages. The only reason it could be different is maybe due to rounding issues, maybe if it handles a lot more numbers, there could be some values that are more inaccurate due to the floating point rounding thats occuring, in great numbers too. Overall though, it seems like the sklearn library is doing the same hting that I am, but it's probably faster, more efficient, and less prone to issues in larger/more diverse data sets.